# Training Notebooks

- [Vesuvius Surface 3D Detection in Keras-JAX](https://www.kaggle.com/code/ipythonx/vesuvius-surface-3d-detection-in-jax)
- [Vesuvius Surface 3D Detection in PyTorch](https://www.kaggle.com/code/ipythonx/vesuvius-surface-3d-detection-in-pytorch)
- [Vesuvius Surface 3D Detection in PyTorch Lightning](https://www.kaggle.com/code/ipythonx/train-vesuvius-surface-3d-detection-in-lightning)
- [[WIP] Vesuvius Surface 2.5D Detection](https://www.kaggle.com/code/ipythonx/wip-vesuvius-surface-2-5d-detection)

**Note**
1. The inference code below is adapted from the **Keras-JAX** version. The PyTorch and Lightning implementations follow the same workflow. Training was performed on a single Tesla T4 (16 GB VRAM) with extended epochs.
2. Both the training and inference pipelines are implemented using [`medicai`](https://github.com/innat/medic-ai), a **Keras 3** based multi-backend medical ML library designed for 2D and 3D classification and segmentation tasks. However, please note, `medicai` project is still new and actively evolving.

# Inference

In [ ]:
var="/kaggle/input/vsdetection-packages-offline-installer-only/whls"
!pip install \
    "$var"/keras_nightly-3.12.0.dev2025100703-py3-none-any.whl \
    "$var"/tifffile-2025.12.20-py3-none-any.whl \
    "$var"/imagecodecs-2026.1.1-cp311-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl \
    "$var"/medicai-0.0.3-py3-none-any.whl \
    --no-index \
    --find-links "$var"

In [ ]:
# --- Protobuf compatibility patch (for old code using MessageFactory.GetPrototype) ---

try:
    from google.protobuf import message_factory as _message_factory

    # Only patch if the method is missing (protobuf >= 5)
    if not hasattr(_message_factory.MessageFactory, "GetPrototype"):
        from google.protobuf.message_factory import GetMessageClass

        def _GetPrototype(self, descriptor):
            # Old API used MessageFactory().GetPrototype(descriptor)
            # New API is GetMessageClass(descriptor). We just bridge them.
            return GetMessageClass(descriptor)

        _message_factory.MessageFactory.GetPrototype = _GetPrototype
        print("Patched protobuf: added MessageFactory.GetPrototype")
    else:
        print("protobuf already has MessageFactory.GetPrototype; no patch needed.")
except Exception as e:
    print("Could not patch protobuf MessageFactory:", e)

import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
from medicai.transforms import (
    Compose,
    ScaleIntensityRange,
)
from medicai.models import SegFormer, TransUNet
from medicai.utils.inference import SlidingWindowInference

import numpy as np
import pandas as pd
import zipfile
import tifffile
from matplotlib import pyplot as plt

keras.config.backend(), keras.version()

**Dataset**

In [ ]:
root_dir = "/kaggle/input/vesuvius-challenge-surface-detection"
test_dir = f"{root_dir}/test_images"
output_dir = "/kaggle/working/submission_masks"
zip_path = "/kaggle/working/submission.zip"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
test_df = pd.read_csv(f"{root_dir}/test.csv")
test_df.head()

**Transformation**

In [ ]:
def val_transformation(image):
    data = {"image": image}
    pipeline = Compose([
        ScaleIntensityRange(
            keys=["image"],
            a_min = 0,
            a_max = 255,
            b_min = 0,
            b_max = 1,
            clip = True,
        ),
    ])
    result = pipeline(data)
    return result["image"]

**Model**

In [ ]:
num_classes=3

def get_model():
    ## LB: 0.486
    # model = SegFormer(
    #     input_shape=(128, 128, 128, 1),
    #     encoder_name='mit_b2',
    #     classifier_activation='softmax',
    #     num_classes=num_classes,
    # )
    # model.load_weights(
    #     "/kaggle/input/vsd-model/keras/segformer.mit.b2/2/segformer.mit.b2.weights.h5"
    # )

    ## LB: 0.5 
    model = TransUNet(
        input_shape=(160, 160, 160, 1),
        encoder_name='seresnext50',
        classifier_activation='softmax',
        num_classes=num_classes,
    )
    model.load_weights(
        "/kaggle/input/opentransunet/pytorch/default/1/model.weights.h5"
    )
    return model

In [ ]:
model = get_model()
model.count_params() / 1e6
# predictor = tf.function(model, jit_compile=True)


In [ ]:
model.instance_describe()

In [ ]:
model

**Sliding Window Inference**

In [ ]:
pred = SlidingWindowInference(
    model,
    roi_size=(160,160,160),
    num_classes = 3,
    mode="gaussian",
    overlap=0.6,
    sw_batch_size = 1
)



In [ ]:
import os
import numpy as np
import tensorflow as tf
from skimage.morphology import skeletonize
from scipy.ndimage import binary_dilation
from scipy import ndimage

def frangi_filter_3d(image, sigmas=(1, 3, 5), beta1=0.5, beta2=15, gamma=None, black_ridges=True):
    """
    Frangi滤波器 - 用于增强3D图像中的管状结构（如血管）。
    
    Frangi滤波器基于Hessian矩阵的特征值来检测管状结构。它通过分析局部二阶导数
    来识别具有"管状"几何特征的区域。
    
    原理：
    1. 计算图像在不同尺度下的Hessian矩阵
    2. 计算Hessian矩阵的特征值（λ1, λ2, λ3）
    3. 根据特征值的关系判断是否为管状结构
    4. 使用Frangi响应函数计算增强值
    
    参数:
        image: numpy数组，形状为 (D, H, W) 或 (D, H, W, 1) 的3D图像
        sigmas: tuple，高斯核的标准差范围，用于多尺度检测
                例如 (1, 3, 5) 表示检测1到5像素宽度的血管
        beta1: float，控制对偏离管状结构的敏感度（默认0.5）
        beta2: float，控制对背景的敏感度（默认15）
        gamma: float，归一化因子，如果为None则自动计算
        black_ridges: bool，True表示检测暗色管状结构（血管），False表示检测亮色结构
    
    返回:
        enhanced: numpy数组，增强后的图像，形状与输入相同
    """
    # 确保输入是3D数组
    if len(image.shape) == 4:
        image = image[..., 0]
    
    # 确保是float类型
    image = image.astype(np.float64)
    
    # 如果检测暗色结构，反转图像
    if black_ridges:
        image = -image
    
    # 初始化输出
    enhanced = np.zeros_like(image)
    
    # 对每个尺度计算Frangi响应
    for sigma in sigmas:
        # 计算Hessian矩阵的各个分量
        # 使用高斯滤波的导数
        hxx = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=2), 
                sigma, axis=1, order=0), 
            sigma, axis=0, order=0)
        
        hyy = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=0), 
                sigma, axis=1, order=2), 
            sigma, axis=0, order=0)
        
        hzz = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=0), 
                sigma, axis=1, order=0), 
            sigma, axis=0, order=2)
        
        hxy = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=1), 
                sigma, axis=1, order=1), 
            sigma, axis=0, order=0)
        
        hxz = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=1), 
                sigma, axis=1, order=0), 
            sigma, axis=0, order=1)
        
        hyz = ndimage.gaussian_filter1d(
            ndimage.gaussian_filter1d(
                ndimage.gaussian_filter1d(image, sigma, axis=2, order=0), 
                sigma, axis=1, order=1), 
            sigma, axis=0, order=1)
        
        # 构建Hessian矩阵并计算特征值
        # 对于3D，我们需要在每个体素处计算3x3矩阵的特征值
        # 这里使用近似方法：计算特征值的平方和
        # 更精确的方法需要逐像素计算特征值，但计算量大
        
        # 简化的Frangi响应计算（基于Hessian矩阵的迹和行列式）
        # 对于管状结构：一个特征值接近0，另外两个较大且符号相同
        trace = hxx + hyy + hzz
        det = (hxx * hyy * hzz + 
               2 * hxy * hxz * hyz - 
               hxx * hyz * hyz - 
               hyy * hxz * hxz - 
               hzz * hxy * hxy)
        
        # Frangi响应函数
        # 对于管状结构，我们希望：
        # - 一个特征值接近0（沿管轴方向）
        # - 另外两个特征值较大且同号（垂直于管轴）
        
        # 使用迹和行列式的组合来近似
        # 归一化
        if gamma is None:
            gamma = np.max(np.abs(trace))
        
        # 计算响应
        # 简化版本：基于Hessian矩阵的Frobenius范数
        hessian_norm = np.sqrt(hxx**2 + hyy**2 + hzz**2 + 
                              2 * (hxy**2 + hxz**2 + hyz**2))
        
        # Frangi响应
        response = np.exp(-beta1 * (trace**2) / (hessian_norm + 1e-10)) * \
                   (1 - np.exp(-beta2 * hessian_norm**2 / (gamma**2 + 1e-10)))
        
        # 只保留正值（管状结构）
        response = np.maximum(response, 0)
        
        # 取所有尺度中的最大值
        enhanced = np.maximum(enhanced, response)
    
    # 归一化到[0, 1]
    if enhanced.max() > 0:
        enhanced = enhanced / enhanced.max()
    
    return enhanced.astype(np.float32)


def apply_frangi_postprocessing(prediction, threshold=0.5, sigmas=(1, 3, 5), 
                                beta1=0.5, beta2=15, use_frangi=True):
    """
    使用Frangi滤波器对分割预测结果进行后处理。
    
    这个方法可以：
    1. 使用Frangi滤波器增强预测结果中的管状结构
    2. 结合原始预测和Frangi增强结果
    3. 应用阈值得到最终的分割掩码
    
    参数:
        prediction: numpy数组，模型预测的概率图，形状为 (D, H, W) 或 (D, H, W, num_classes)
        threshold: float，二值化阈值（默认0.5）
        sigmas: tuple，Frangi滤波器的尺度参数
        beta1: float，Frangi参数
        beta2: float，Frangi参数
        use_frangi: bool，是否使用Frangi滤波器（如果False，只做阈值化）
    
    返回:
        processed_mask: numpy数组，处理后的二值掩码，形状为 (D, H, W)
    """
    # 处理多类别预测
    if len(prediction.shape) == 4:
        # 如果是多类别，取前景类别（假设类别1是血管/墨水）
        if prediction.shape[-1] > 1:
            pred_prob = prediction[..., 1]  # 取类别1的概率
        else:
            pred_prob = prediction[..., 0]
    else:
        pred_prob = prediction
    
    if use_frangi:
        # 应用Frangi滤波器增强管状结构
        frangi_enhanced = frangi_filter_3d(
            pred_prob, 
            sigmas=sigmas, 
            beta1=beta1, 
            beta2=beta2,
            black_ridges=True  # 假设检测暗色结构
        )
        
        # 结合原始预测和Frangi增强结果
        # 方法1：加权平均
        combined = 0.7 * pred_prob + 0.3 * frangi_enhanced
        
        # 方法2：取最大值（保留更强的响应）
        # combined = np.maximum(pred_prob, frangi_enhanced)
        
        # 方法3：只在Frangi检测到的区域增强
        # frangi_mask = frangi_enhanced > 0.3
        # combined = pred_prob.copy()
        # combined[frangi_mask] = np.maximum(pred_prob[frangi_mask], frangi_enhanced[frangi_mask])
        
        processed_prob = combined
    else:
        processed_prob = pred_prob
    
    # 二值化
    processed_mask = (processed_prob > threshold).astype(np.uint8)
    
    return processed_mask

def build_anisotropic_struct(z_radius, xy_radius):
    """
    构建各向异性结构元素用于3D闭运算
    
    参数:
        z_radius: z方向的半径
        xy_radius: xy平面的半径
    
    返回:
        structure: 3D结构元素，如果半径都为0则返回None
    """
    if z_radius == 0 and xy_radius == 0:
        return None
    
    size_z = 2 * z_radius + 1
    size_xy = 2 * xy_radius + 1
    
    struct = np.zeros((size_z, size_xy, size_xy), dtype=bool)
    center_z = z_radius
    center_xy = xy_radius
    
    # 创建各向异性结构
    for z in range(size_z):
        for y in range(size_xy):
            for x in range(size_xy):
                dz = abs(z - center_z)
                dxy = np.sqrt((x - center_xy)**2 + (y - center_xy)**2)
                if dz <= z_radius and dxy <= xy_radius:
                    struct[z, y, x] = True
    
    return struct

In [ ]:
import numpy as np
import tifffile
import scipy.ndimage as ndi
from skimage.morphology import remove_small_objects
from scipy.ndimage import generate_binary_structure
from skimage.morphology import remove_small_objects

def load_volume(path):
    vol = tifffile.imread(path)          # (D, H, W)
    vol = vol.astype(np.float32)
    vol = vol[None, ..., None]           # (1, D, H, W, 1)
    return vol


# ==========================================
# ROTATION TTA HELPERS (CLOCKWISE)
# ==========================================
def rot90_volume(vol, k):
    """
    Rotate volume k times 90° clockwise in HW plane.
    vol:
      (1, D, H, W, 1) OR (D, H, W)
    """
    if vol.ndim == 5:
        return np.rot90(vol, k=-k, axes=(2, 3))
    else:
        return np.rot90(vol, k=-k, axes=(1, 2))


def unrot90_volume(vol, k):
    return rot90_volume(vol, (4 - k) % 4)


def predict_probs_tta_rot(sample):
    """
    4x rotation TTA: 0°, 90°, 180°, 270°
    sample: (1, D, H, W, 1)
    returns: averaged probs (D, H, W)
    """
    probs_accum = []

    for k in range(4):
        s_rot = rot90_volume(sample, k)

        out = pred(s_rot)              # (1, D, H, W, 2)
        out = np.asarray(out)
        probs = out[0, ..., 1]         # (D, H, W)

        probs = unrot90_volume(probs, k)
        probs_accum.append(probs)

    return np.mean(probs_accum, axis=0)


# ==========================================
# HELPER: Anisotropic Structure Builder
# ==========================================
def build_anisotropic_struct(z_radius: int, xy_radius: int):
    z, r = z_radius, xy_radius

    if z == 0 and r == 0:
        return None

    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct

    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct

    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct


# ==========================================
# MAIN POST-PROCESSING LOGIC
# ==========================================
def topo_postprocess(
    probs,          # (D, H, W)
    T_low=0.90,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
):
    # --- Step 1: 3D Hysteresis ---
    strong = probs >= T_high
    weak   = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # --- Step 2: 3D Anisotropic Closing ---
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # --- Step 3: Dust Removal ---
    if dust_min_size > 0:
        mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)


def topo_postprocess_with_frangi(
    probs,          # (D, H, W)
    T_low=0.90,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
    # Frangi参数
    use_frangi=True,
    frangi_weight=0.3,      # Frangi增强的权重 (0-1)
    frangi_sigmas=(1, 3, 5),
    frangi_beta1=0.5,
    frangi_beta2=15,
    frangi_mode='enhance_before',  # 'enhance_before', 'enhance_after', 'replace_hyst'
):
    """
    集成Frangi滤波器的拓扑后处理
    
    参数:
        probs: 概率图 (D, H, W)
        T_low, T_high: Hysteresis阈值
        z_radius, xy_radius: 各向异性闭运算参数
        dust_min_size: 小对象去除的最小尺寸
        use_frangi: 是否使用Frangi滤波器
        frangi_weight: Frangi增强的权重（0-1之间）
        frangi_sigmas: Frangi滤波器的尺度参数
        frangi_beta1, frangi_beta2: Frangi参数
        frangi_mode: Frangi集成模式
            - 'enhance_before': 在Hysteresis之前增强概率图（推荐）
            - 'enhance_after': 在Hysteresis之后增强掩码
            - 'replace_hyst': 用Frangi增强的概率图替代Hysteresis
    
    返回:
        mask: 处理后的二值掩码 (D, H, W) uint8
    """
    if not use_frangi:
        # 如果不使用Frangi，直接使用原始后处理
        return topo_postprocess(
            probs, T_low, T_high, z_radius, xy_radius, dust_min_size
        )
    
    # 应用Frangi滤波器增强
    frangi_enhanced = frangi_filter_3d(
        probs,
        sigmas=frangi_sigmas,
        beta1=frangi_beta1,
        beta2=frangi_beta2,
        black_ridges=True  # 假设检测暗色结构（血管）
    )
    
    if frangi_mode == 'enhance_before':
        # 模式1: 在Hysteresis之前增强概率图（推荐）
        # 结合原始概率和Frangi增强结果
        enhanced_probs = (1 - frangi_weight) * probs + frangi_weight * frangi_enhanced
        
        # 然后进行标准的拓扑后处理
        return topo_postprocess(
            enhanced_probs, T_low, T_high, z_radius, xy_radius, dust_min_size
        )
    
    elif frangi_mode == 'enhance_after':
        # 模式2: 在Hysteresis之后增强掩码
        # 先进行标准拓扑后处理
        mask = topo_postprocess(
            probs, T_low, T_high, z_radius, xy_radius, dust_min_size
        )
        
        # 将掩码转换为概率图（用于Frangi）
        mask_probs = mask.astype(np.float32)
        
        # 应用Frangi增强
        frangi_mask = frangi_filter_3d(
            mask_probs,
            sigmas=frangi_sigmas,
            beta1=frangi_beta1,
            beta2=frangi_beta2,
            black_ridges=True
        )
        
        # 结合原始掩码和Frangi增强结果
        enhanced_mask = np.maximum(
            mask_probs,
            frangi_mask * frangi_weight
        )
        
        # 二值化
        return (enhanced_mask > 0.5).astype(np.uint8)
    
    elif frangi_mode == 'replace_hyst':
        # 模式3: 用Frangi增强的概率图替代Hysteresis
        # 结合原始概率和Frangi增强结果
        enhanced_probs = (1 - frangi_weight) * probs + frangi_weight * frangi_enhanced
        
        # 使用简单的阈值化替代Hysteresis
        mask = enhanced_probs >= T_high
        
        if not mask.any():
            return np.zeros_like(probs, dtype=np.uint8)
        
        # 然后进行闭运算和小对象去除
        if z_radius > 0 or xy_radius > 0:
            struct_close = build_anisotropic_struct(z_radius, xy_radius)
            if struct_close is not None:
                mask = ndi.binary_closing(mask, structure=struct_close)
        
        if dust_min_size > 0:
            mask = remove_small_objects(mask.astype(bool), min_size=dust_min_size)
        
        return mask.astype(np.uint8)
    
    else:
        raise ValueError(f"Unknown frangi_mode: {frangi_mode}")

# ==========================================
# PREDICT (WITH ROTATION TTA)
# ==========================================
# def predict(
#     sample,
#     iid=None,
#     T_low=0.50,
#     T_high=0.90,
#     z_radius=1,
#     xy_radius=0,
#     dust_min_size=100,
# ):
#     """
#     sample: (1, D, H, W, 1)
#     """

#     # --------- ROTATION TTA PROBS ---------
#     probs_fg = predict_probs_tta_rot(sample)   # (D, H, W)

#     if iid is not None:
#         np.save(iid, probs_fg)

#     # --------- POSTPROCESS (UNCHANGED) ----
#     final = topo_postprocess(
#         probs_fg,
#         T_low=T_low,
#         T_high=T_high,
#         z_radius=z_radius,
#         xy_radius=xy_radius,
#         dust_min_size=dust_min_size,
#     )

#     return final  # (D, H, W) uint8 {0,1}

def predict(
    sample,  # TTA预测函数
    iid=None,
    T_low=0.50,
    T_high=0.90,
    z_radius=1,
    xy_radius=0,
    dust_min_size=100,
    # Frangi参数
    use_frangi=True,
    frangi_weight=0.3,
    frangi_sigmas=(1, 3, 5),
    frangi_beta1=0.5,
    frangi_beta2=15,
    frangi_mode='enhance_after',
):
    """
    完整的预测函数，集成Frangi滤波器
    
    参数:
        sample: 输入样本 (1, D, H, W, 1)
        predict_probs_tta_rot_func: TTA预测函数
        iid: 保存概率图的路径（可选）
        其他参数: 后处理参数
    
    返回:
        final: 最终掩码 (D, H, W) uint8 {0,1}
    """
    # --------- ROTATION TTA PROBS ---------
    probs_fg = predict_probs_tta_rot(sample)   # (D, H, W)

    if iid is not None:
        np.save(iid, probs_fg)

    # --------- POSTPROCESS WITH FRANGI ----
    final = topo_postprocess_with_frangi(
        probs_fg,
        T_low=T_low,
        T_high=T_high,
        z_radius=z_radius,
        xy_radius=xy_radius,
        dust_min_size=dust_min_size,
        use_frangi=use_frangi,
        frangi_weight=frangi_weight,
        frangi_sigmas=frangi_sigmas,
        frangi_beta1=frangi_beta1,
        frangi_beta2=frangi_beta2,
        frangi_mode=frangi_mode,
    )

    return final  # (D, H, W) uint8 {0,1}



**Prediction and Zip Submission**

In [ ]:
testing = False

In [ ]:
if testing:
    
    test_dir = "/kaggle/input/vesuvius-challenge-surface-detection/train_images"
    test_df = pd.read_csv(f"{root_dir}/train.csv")
    test_ids = {956073442, 961304774,969293709,975031774,985841575,992852942}
    test_df = (
        test_df
        .loc[test_df["id"].isin(test_ids)]
        .reset_index(drop=True)
    )

In [ ]:
with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for image_id in test_df["id"]:
        tif_path = f"{test_dir}/{image_id}.tif"
            
        volume = load_volume(tif_path)
        volume = val_transformation(volume)
        if testing :
            output = predict(volume,f"{image_id}") 
        else :
            output = predict(volume)
        
        out_path = f"{output_dir}/{image_id}.tif"
        tifffile.imwrite(out_path, output.astype(np.uint8))

        z.write(out_path, arcname=f"{image_id}.tif")
        os.remove(out_path)

print("Submission ZIP:", zip_path)